# 4ft-Miner: Reliable guests

**Author:** Pavlo Romanovyč Prytula

We look for guest characteristics that make a booking very unlikely to be canceled.
The task focuses on the behaviour and history of the guest, not on the parameters of the booking.
Only attributes describing the guest are used. `needs_parking` is left out because it is recorded
almost only for guests who arrived (leakage), `previous_cancellations_cat` because of an anomaly
in group bookings, and booking parameters (`customer_type`, `deposit_type`, channel, stay) because
they do not describe the guest.

**Question:** Which signs of guest engagement and loyalty (special requests, repeated stay,
earlier stays that were not canceled) go together with bookings that are not canceled?
Does this differ between the City Hotel and the Resort Hotel?

**Hypothesis:** Guests with at least one earlier stay that was not canceled keep their booking
at least 30 % more often (relative increase) than an average guest of the same hotel.

| Part | Setting |
| --- | --- |
| Antecedent (A) | `special_requests_cat` (subset, 1), `is_repeated_guest = 1`, `previous_bookings_not_canceled_cat = 1+` |
| Antecedent length | 1–3 |
| Succedent (S) | `is_canceled = 0` (the stay was NOT canceled) |
| Condition (C) | `hotel` (subset, 1) |
| Quantifiers | `aad ≥ 0.3`, `Base ≥ 1000` |

In [1]:
import pandas as pd
from cleverminer import cleverminer

df = pd.read_csv("../../data/processed/hotel_bookings_categorized.csv", dtype=str)
df = df[[
    "hotel",
    "is_canceled",
    "special_requests_cat",
    "is_repeated_guest",
    "previous_bookings_not_canceled_cat",
]]

print("Bookings:", len(df))

Cleverminer version  1.2.8
Bookings: 118558


## Share of not canceled bookings

Share of bookings that were not canceled, separately for both hotels.
This is the baseline against which `aad` is computed, because the condition splits the data by hotel.

In [2]:
not_canceled = df["is_canceled"] == "0"

display(not_canceled.groupby(df["hotel"]).mean().round(3))

for column in ["special_requests_cat", "is_repeated_guest", "previous_bookings_not_canceled_cat"]:
    display(pd.crosstab(df[column], df["hotel"], values=not_canceled, aggfunc="mean").round(3))

hotel
City Hotel      0.581
Resort Hotel    0.720
Name: is_canceled, dtype: float64

hotel,City Hotel,Resort Hotel
special_requests_cat,,
0,0.448,0.674
1,0.779,0.779
2+,0.794,0.776


hotel,City Hotel,Resort Hotel
is_repeated_guest,,
0,0.577,0.711
1,0.761,0.933


hotel,City Hotel,Resort Hotel
previous_bookings_not_canceled_cat,,
0,0.574,0.707
1+,0.924,0.961


## 4ft-Miner

The condition `hotel` evaluates every rule separately in the City Hotel and in the Resort Hotel.
The two hotels have different baselines (58.1 % vs. 72.0 % of bookings not canceled),
so `aad` always compares a group with the average of its own hotel.
`aad ≥ 0.3` requires the group to keep its bookings at least 30 % more often than this average,
i.e. at least 75.5 % in the City Hotel and at least 93.6 % in the Resort Hotel.
The threshold is therefore stricter in the Resort Hotel, where `aad` cannot exceed 0.39
(even if 100 % of the bookings are kept).
`Base ≥ 1000` requires at least 1000 not canceled bookings in the group, so small combinations
of characteristics are removed.
Binary attributes use only the value `1` (or `1+`), so the rules describe the presence of a characteristic.

In [3]:
miner = cleverminer(
    df=df,
    proc="4ftMiner",
    quantifiers={
        "aad": 0.3,
        "Base": 1000,
    },
    ante={
        "attributes": [
            {"name": "special_requests_cat", "type": "subset", "minlen": 1, "maxlen": 1},
            {"name": "is_repeated_guest", "type": "one", "value": "1"},
            {"name": "previous_bookings_not_canceled_cat", "type": "one", "value": "1+"},
        ],
        "minlen": 1,
        "maxlen": 3,
        "type": "con",
    },
    succ={
        "attributes": [
            {"name": "is_canceled", "type": "one", "value": "0"},
        ],
        "minlen": 1,
        "maxlen": 1,
        "type": "con",
    },
    cond={
        "attributes": [
            {"name": "hotel", "type": "subset", "minlen": 1, "maxlen": 1},
        ],
        "minlen": 1,
        "maxlen": 1,
        "type": "con",
    },
    opts={
        "no_automatic_data_conversions": True,
        "disable_progressbar": True,
    },
)

Cleverminer version 1.2.8.
Progressbar will not be shown.
No automatic data conversions will be made.


Progressbar will not be shown.
No automatic data conversions will be made.
Starting data preparation ...
Encoding columns into bit-form...
Encoding columns into bit-form...done
Data preparation finished.
Progressbar will not be shown.
No automatic data conversions will be made.
Will go for  4ftMiner
Starting to mine rules.
Done. Total verifications : 13, rules 8, times: prep 0.08sec, processing 0.01sec


## Results

Overview of all rules. `BASE` is the number of not canceled bookings of the group,
`CONF` is the share of not canceled bookings in the group and `AAD` is the relative increase
compared with the average of the hotel.

In [4]:
miner.print_summary()
miner.print_rulelist()


CleverMiner task processing summary:

Task type : 4ftMiner
Number of verifications : 13
Number of rules : 8
Total time needed : 00h 00m 00s
Time of data preparation : 00h 00m 00s
Time of rule mining : 00h 00m 00s


List of rules:
RULEID BASE  CONF  AAD    Rule
     1 16584 0.779 +0.341 special_requests_cat(1) => is_canceled(0) | hotel(City Hotel)
     2  7871 0.794 +0.367 special_requests_cat(2+) => is_canceled(0) | hotel(City Hotel)
     3  1394 0.761 +0.310 is_repeated_guest(1) => is_canceled(0) | hotel(City Hotel)
     4  1296 0.924 +0.591 is_repeated_guest(1) & previous_bookings_not_canceled_cat(1+) => is_canceled(0) | hotel(City Hotel)
     5  1453 0.924 +0.590 previous_bookings_not_canceled_cat(1+) => is_canceled(0) | hotel(City Hotel)
     6  1126 0.958 +0.331 special_requests_cat(0) & previous_bookings_not_canceled_cat(1+) => is_canceled(0) | hotel(Resort Hotel)
     7  1355 0.964 +0.340 is_repeated_guest(1) & previous_bookings_not_canceled_cat(1+) => is_canceled(0) | hotel(Re

### All rules in detail

Fourfold tables of all rules, one below another. The table is computed within the condition (hotel).

In [5]:
for rule_id in range(1, miner.get_rulecount() + 1):
    miner.print_rule(rule_id)



Rule id : 1

Base : 16584  Relative base : 0.140  CONF : 0.779  AAD : +0.341  BAD : -0.341

Cedents:
  antecedent : special_requests_cat(1)
  succcedent : is_canceled(0)
  condition  : hotel(City Hotel)

Fourfold table
    |  S  |  ¬S |
----|-----|-----|
 A  |16584| 4711|
----|-----|-----|
¬A  |29249|28350|
----|-----|-----|



Rule id : 2

Base :  7871  Relative base : 0.066  CONF : 0.794  AAD : +0.367  BAD : -0.367

Cedents:
  antecedent : special_requests_cat(2+)
  succcedent : is_canceled(0)
  condition  : hotel(City Hotel)

Fourfold table
    |  S  |  ¬S |
----|-----|-----|
 A  | 7871| 2038|
----|-----|-----|
¬A  |37962|31023|
----|-----|-----|



Rule id : 3

Base :  1394  Relative base : 0.012  CONF : 0.761  AAD : +0.310  BAD : -0.310

Cedents:
  antecedent : is_repeated_guest(1)
  succcedent : is_canceled(0)
  condition  : hotel(City Hotel)

Fourfold table
    |  S  |  ¬S |
----|-----|-----|
 A  | 1394|  438|
----|-----|-----|
¬A  |44439|32623|
----|-----|-----|



Rule id : 

## Interpretation

- **Earlier stays that were not canceled** are the strongest signal. In the City Hotel 92.4 % of such
  bookings are kept (rule 5, +59 % against the hotel average of 58.1 %), in the Resort Hotel 96.1 %
  (rule 8, +33 % against 72.0 %). The hypothesis holds in both hotels.
- **Special requests** matter only in the City Hotel: with 1 request 77.9 % of bookings are kept
  (rule 1), with 2+ requests 79.4 % (rule 2). These rules cover the largest groups (16 584 and
  7871 kept bookings). In the Resort Hotel only rules with earlier stays pass the stricter threshold.
- `is_repeated_guest` and `previous_bookings_not_canceled_cat` overlap strongly, so rules with both
  attributes (rules 4 and 7) bring little new information.
- **Not a random pattern:** the rules cover only about 1 % of all bookings, but the effects are large.
  For example, rule 5 covers 1573 bookings. If these guests behaved like an average City Hotel guest,
  about 914 of them would be kept; in reality 1453 are kept. A difference this large cannot be
  explained by chance. The small share limits the practical impact of the rules, not their reliability.

These are associations, not causal effects.